In [1]:
import sys
print("Python executable:", sys.executable)

import re
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

print("All imports successful")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice available:", device)
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("No GPU detected — training will run on CPU, which will be significantly slower.")

Python executable: c:\Users\sehba\OneDrive - Asia Pacific University of Technology And Innovation (APU)\Documents\fraud-detector-fyp\backend\venv\Scripts\python.exe
All imports successful

Device available: cpu
No GPU detected — training will run on CPU, which will be significantly slower.


In [2]:
import time

tokenizer_bench = BertTokenizerFast.from_pretrained("bert-base-uncased")
model_bench = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model_bench.to(device)
model_bench.train()

batch_size = 8
seq_len = 512
dummy_input_ids = torch.randint(0, tokenizer_bench.vocab_size, (batch_size, seq_len)).to(device)
dummy_attention_mask = torch.ones((batch_size, seq_len), dtype=torch.long).to(device)
dummy_labels = torch.randint(0, 2, (batch_size,)).to(device)

optimizer = torch.optim.AdamW(model_bench.parameters(), lr=2e-5)

# Warm-up pass, not timed - first pass includes one-time setup overhead
outputs = model_bench(input_ids=dummy_input_ids, attention_mask=dummy_attention_mask, labels=dummy_labels)
outputs.loss.backward()
optimizer.step()
optimizer.zero_grad()

# Timed passes
n_trials = 3
times = []
for _ in range(n_trials):
    start = time.time()
    outputs = model_bench(input_ids=dummy_input_ids, attention_mask=dummy_attention_mask, labels=dummy_labels)
    outputs.loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    times.append(time.time() - start)

avg_batch_time = sum(times) / len(times)
print(f"Average time per batch (batch_size={batch_size}, seq_len={seq_len}): {avg_batch_time:.2f} seconds")

n_train_rows = 12516
batches_per_epoch = n_train_rows / batch_size
est_epoch_time_min = (batches_per_epoch * avg_batch_time) / 60

print(f"Estimated batches per epoch: {batches_per_epoch:.0f}")
print(f"Estimated time per epoch: {est_epoch_time_min:.1f} minutes")
print(f"Estimated time for 3 epochs: {est_epoch_time_min * 3:.1f} minutes ({est_epoch_time_min * 3 / 60:.1f} hours)")

del model_bench

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\sehba\OneDrive - Asia Pacific University of Technology And Innovation (APU)\Documents\fraud-detector-fyp\backend\venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sehba\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Average time per batch (batch_size=8, seq_len=512): 29.63 seconds
Estimated batches per epoch: 1564
Estimated time per epoch: 772.6 minutes
Estimated time for 3 epochs: 2317.8 minutes (38.6 hours)
